In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from config import config

import importlib
import src.preprocessing as prep

importlib.reload(prep)

<module 'src.preprocessing' from 'c:\\Users\\aukha\\Python Practicum\\Titanic_kaggle\\src\\preprocessing.py'>

In [2]:

from src.data import load_data

train_df, test_df = load_data(config)






In [3]:
submission_12 = pd.read_csv('./outputs/submission_12.csv')
submission_13 = pd.read_csv('./outputs/submission_13.csv')

changed = (
    submission_12['Survived']
    != submission_13['Survived']
).sum()

print(changed)

2


In [4]:
comparison = submission_12.copy()

comparison = comparison.rename(
    columns={'Survived': 'prediction_12'}
)

comparison['prediction_13'] = submission_13['Survived']

changed = comparison[
    comparison['prediction_12']
    != comparison['prediction_13']
]

print(changed)

     PassengerId  prediction_12  prediction_13
376         1268              0              1
390         1282              1              0


In [7]:
cols = [
    'PassengerId',
    'Pclass',
    'Sex',
    'Age',
    'SibSp',
    'Parch',
    'Fare',
    'Embarked',
    # 'Title',
    # 'FamilySize',
    # 'IsAlone',
]

print(
    test_df[
        test_df['PassengerId'].isin([1268, 1282])
    ][cols]
)

     PassengerId  Pclass     Sex   Age  SibSp  Parch     Fare Embarked
376         1268       3  female  22.0      2      0   8.6625        S
390         1282       1    male  23.0      0      0  93.5000        S


In [4]:
print(
    train_df
    .groupby('SibSp')['Survived']
    .mean()
)

SibSp
0    0.345395
1    0.535885
2    0.464286
3    0.250000
4    0.166667
5    0.000000
8    0.000000
Name: Survived, dtype: float64


In [ ]:
import src.preprocessing as prep

train_df = prep.map_titles(train_df)

print(train_df["Title"].value_counts())

In [ ]:
sorted(train_df["Title"].unique())

In [ ]:
train_df, test_df = load_data(config)

print(train_df['Age'].isna().sum())
print(test_df['Age'].isna().sum())

train_df, test_df = prep.preprocess_age(
    train_df=train_df,
    test_df=test_df
)

print(train_df['Age'].isna().sum())
print(test_df['Age'].isna().sum())

In [ ]:
age_before, _ = load_data(config)

age_after, _ = prep.preprocess_age(
    train_df=age_before,
    test_df=age_before
)

check_df = age_after.loc[
    age_before['Age'].isna(),
    ['Name', 'Title', 'Age']
].copy()

check_df = check_df.rename(
    columns={'Age': 'Age_filled'}
)

print(check_df.head(50))

In [ ]:
age_before, _ = load_data(config)

age_before = prep.extract_title(age_before)
age_before = prep.map_titles(age_before)

age_before.groupby('Title')['Age'].agg(
    ['count', 'mean', 'median']
)

In [ ]:
f, ax = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(age_before['Age'], bins=30, kde=True, ax=ax[0])
ax[0].set_title('Age before imputation')

sns.histplot(age_after['Age'], bins=30, kde=True, ax=ax[1])
ax[1].set_title('Age after imputation')

plt.show()

In [ ]:
from src.data import load_data
from src.preprocessing import preprocess_age

train_df, kaggle_test_df = load_data(config)

train_cv_df, holdout_df = train_test_split(
    train_df,
    test_size=config.split.test_size,
    random_state=config.general.seed,
    shuffle=config.dataloader_params.shuffle,
    stratify=train_df['Survived']
)

train_cv_df, holdout_df = preprocess_age(
    train_df=train_cv_df,
    test_df=holdout_df
)

_, kaggle_test_df = preprocess_age(
    train_df=train_cv_df,
    test_df=kaggle_test_df
)

print(train_cv_df['Age'].isna().sum())
print(holdout_df['Age'].isna().sum())
print(kaggle_test_df['Age'].isna().sum())

Ниже черрновик для catboost

In [1]:
from sklearn.model_selection import train_test_split

from config import config
from src.data import load_data
from src.train_functions import build_pipeline


train_df, _ = load_data(config)

features = train_df.drop(columns=['Survived'])
labels = train_df['Survived']

features_train, _, labels_train, _ = train_test_split(
    features,
    labels,
    test_size=config.split.test_size,
    random_state=config.general.seed,
    shuffle=config.dataloader_params.shuffle,
    stratify=labels,
)

pipe = build_pipeline(config)

features_transformed = (
    pipe[:-1]
    .fit_transform(
        features_train,
        labels_train,
    )
)

features_transformed.head()

,Age,Fare,FamilySize,IsAlone,CabinKnown,Pclass,Sex,Title,Deck,Embarked
849,35.0,89.1042,2,0,1,1,female,Mrs,C,C
261,3.0,31.3875,7,0,0,3,male,Master,Unknown,S
752,33.0,9.5000,1,1,0,3,male,Mr,Unknown,S
537,30.0,106.4250,1,1,0,1,female,Miss,Unknown,C
432,42.0,26.0000,2,0,0,2,female,Mrs,Unknown,S


In [2]:
features_transformed.dtypes

Age           float64
Fare          float64
FamilySize      int64
IsAlone         int64
CabinKnown      int64
Pclass            str
Sex               str
Title             str
Deck              str
Embarked          str
dtype: object

In [3]:
features_transformed.isna().sum()

Age           0
Fare          0
FamilySize    0
IsAlone       0
CabinKnown    0
Pclass        0
Sex           0
Title         0
Deck          0
Embarked      0
dtype: int64